# DNA Splice Junction Classification
## Data Cleaning

In the previous notebook, I examined the structure and quality of the raw dataset.

I found no missing values, invalid sequence characters, inconsistent sequence lengths, or formatting issues. However, I identified duplicate observations, repeated DNA sequences, and one sequence associated with two different class labels.

In this notebook, I will investigate these issues and make only the cleaning decisions that are justified by the data.

## 1. Why Data Cleaning?

Data cleaning prepares the dataset for feature engineering and model training.

For this project, I do not want to remove observations simply because they look unusual. Each cleaning decision should have a clear reason and should preserve the biological information in the original dataset where possible.

I will focus on the issues identified during dataset understanding:
- duplicate observations
- repeated DNA sequences
- the conflicting sequence label

In [1]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("../data/raw/splice.data")

raw_data = pd.read_csv(
    DATA_PATH,
    header=None,
    skipinitialspace=True
)

raw_data.columns = ["class", "instance_name", "sequence"]

raw_data.head()

,class,instance_name,sequence
0,EI,ATRINS-DONOR-521,CCAGCTGCATCACAGGAGGCCAGCGAGCAGGTCTGTTCCAAGGGCC...
1,EI,ATRINS-DONOR-905,AGACCCGCCGGGAGGCGGAGGACCTGCAGGGTGAGCCCCACCGCCC...
2,EI,BABAPOE-DONOR-30,GAGGTGAAGGACGTCCTTCCCCAGGAGCCGGTGAGAAGCGCAGTCG...
3,EI,BABAPOE-DONOR-867,GGGCTGCGTTGCTGGTCACATTCCTGGCAGGTATGGGGCGGGGCTT...
4,EI,BABAPOE-DONOR-2817,GCTCAGCCCCCAGGTCACCCAGGAACTGACGTGAGTGTCCCCATCC...


### WHY

I will keep the original raw dataset unchanged and create a working copy for cleaning.

This allows me to compare the cleaned data with the original data if needed.

In [3]:
cleaned_data = raw_data.copy()

print("Original shape:", raw_data.shape)
print("Working copy shape:", cleaned_data.shape)

Original shape: (3190, 3)
Working copy shape: (3190, 3)


### WHY

The raw dataset contains 12 completely duplicated rows.

Since these rows contain the same class, instance name, and sequence, keeping multiple copies would unnecessarily duplicate the same observation.

I will remove only these exact duplicates.

In [4]:
before = len(cleaned_data)

cleaned_data = cleaned_data.drop_duplicates()

after = len(cleaned_data)

print("Rows removed:", before - after)
print("Cleaned shape:", cleaned_data.shape)

Rows removed: 12
Cleaned shape: (3178, 3)


### FIND

After removing exact duplicate rows, some DNA sequences still occur in multiple observations.

These are not necessarily duplicates because the same sequence can have different instance names or biological labels.

One sequence was also found with conflicting labels (`IE` and `N`).

### DECIDE

I will not remove repeated DNA sequences or change their labels.

They are retained because removing them based only on sequence repetition could discard potentially meaningful observations.

The conflicting sequence will be explicitly documented as a dataset limitation and considered when designing the model evaluation.

In [5]:
print("Shape:", cleaned_data.shape)
print("Missing values:", cleaned_data.isna().sum().sum())
print("Duplicate complete rows:", cleaned_data.duplicated().sum())

Shape: (3178, 3)
Missing values: 0
Duplicate complete rows: 0


In [6]:
print("Class distribution:")
print(cleaned_data["class"].value_counts())

print("\nSequence lengths:")
print(cleaned_data["sequence"].str.len().value_counts().sort_index())

print("\nUnique sequence characters:")
print(sorted(set("".join(cleaned_data["sequence"]))))

Class distribution:
class
N     1655
IE     762
EI     761
Name: count, dtype: int64

Sequence lengths:
sequence
60    3178
Name: count, dtype: int64

Unique sequence characters:
['A', 'C', 'D', 'G', 'N', 'R', 'S', 'T']


## 1.6 Data Cleaning — Summary

### FIND

The raw dataset contained 3,190 observations.

I removed 12 exact duplicate rows, leaving 3,178 observations.

No missing values or invalid sequence characters required correction.

Repeated DNA sequences were retained because sequence repetition does not necessarily indicate an erroneous observation.

The conflicting sequence with two different class labels was also retained rather than assigning a new label without biological justification.

### DECIDE

The cleaned dataset contains 3,178 observations and is ready for DNA-specific exploratory analysis.

I will use this cleaned dataset for the next stages of the project.

The duplicate-sequence and conflicting-label issue will remain documented as a dataset consideration rather than being artificially corrected.